In [188]:
import numpy as np
import pandas as pd
import pickle

import plotly.express as px
import plotly.graph_objects as go

In [189]:
reference_path = "../../results/transit/reference.parquet"
calibration_path = "../../results/transit/calibration.p"

In [190]:
df_reference = pd.read_parquet(reference_path)
df_reference

,person_id,trip_id,departure_time,weight,legs_bus,legs_rail,legs_subway,legs_tram,transfers,origin_x,origin_y,destination_x,destination_y,request_index
0,0,0,38700.0,29.75996,2,0,0,0,1,841663.76,6518357.22,841867.50,6518592.90,0
1,0,1,42300.0,29.75996,2,0,0,0,1,841835.50,6518625.00,841663.76,6518357.22,1
2,1,2,30000.0,51.85581,1,0,1,1,2,841663.76,6518357.22,842352.10,6520216.90,2
3,1,3,63900.0,51.85581,1,0,1,1,2,842497.94,6520235.16,841663.76,6518357.22,3
4,2,4,26880.0,53.59816,2,0,0,0,1,841663.76,6518357.22,840696.10,6515159.00,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8870,36428,99290,46800.0,82.63741,2,0,0,0,1,847820.45,6525592.74,882879.10,6542587.00,8870
8871,36428,99291,52200.0,82.63741,2,0,0,0,1,882583.20,6543489.00,881747.60,6542957.00,8871
8872,36428,99292,57600.0,82.63741,2,0,0,0,1,880992.00,6543013.00,882867.80,6542574.00,8872
8873,36545,99527,24000.0,80.04115,2,1,0,0,2,849773.52,6522509.70,843024.20,6518704.90,8873


In [191]:
with open(calibration_path, "rb") as f:
    history = pickle.load(f)

In [192]:
df_progress = pd.DataFrame.from_records([
    h["utilities"] for h in history
])

df_progress["objective"] = [h["objective"] for h in history]
df_progress["evaluation"] = np.arange(len(df_progress))

df_progress["best_objective"] = df_progress["objective"].cummin()

In [193]:
figure_scatter = px.scatter(df_progress, x = "evaluation", y = "objective")
figure_line = px.line(df_progress, x = "evaluation", y = "best_objective")
go.Figure(data = figure_scatter.data + figure_line.data)

In [194]:
df_values = df_progress.drop(columns = ["objective", "best_objective"])

df_best = df_progress.copy()
is_best = df_best["objective"] == df_best["best_objective"]
df_best = df_best.drop(columns = ["objective", "best_objective"])
df_best = df_best.set_index("evaluation")
df_best.loc[~is_best, :] = np.nan
df_best = df_best.ffill().reset_index()

df_values = df_values.melt("evaluation")
df_best = df_best.melt("evaluation")

In [195]:
figure_scatter = px.scatter(df_values, x = "evaluation", y = "value", color = "variable")
figure_best = px.line(df_best, x = "evaluation", y = "value", color = "variable")
go.Figure(data = figure_scatter.data + figure_best.data)

In [196]:
best_evaluation = history[df_progress["objective"].argmin()]
df_evaluation = best_evaluation["evaluation"]

df_reference["request_index"] = np.arange(len(df_reference))
df_evaluation = pd.merge(df_evaluation, df_reference[["request_index", "weight"]])

In [197]:
df_initial = pd.merge(history[0]["evaluation"], df_reference[["request_index", "weight"]])

In [198]:
df_reference_transfers = df_reference.groupby("transfers")["weight"].sum().reset_index()
df_reference_transfers["data"] = "reference"

df_evaluation_transfers = df_evaluation.groupby("transfers")["weight"].sum().reset_index()
df_evaluation_transfers["data"] = "calibration"

df_initial_transfers = df_initial.groupby("transfers")["weight"].sum().reset_index()
df_initial_transfers["data"] = "initial"

df_transfers = pd.concat([df_reference_transfers, df_evaluation_transfers, df_initial_transfers])
px.bar(df_transfers, x = "transfers", y = "weight", color = "data", barmode = "group")

In [199]:
modes = ["rail", "subway", "bus", "tram"]
columns = ["legs_{}".format(m) for m in modes]
rename = { c: m for c, m in zip(columns, modes) }

df_reference_modes = df_reference[columns + ["weight"]]
df_reference_modes = df_reference_modes.rename(columns = rename)
for mode in modes: df_reference_modes[mode] *= df_reference_modes["weight"].values
df_reference_modes = df_reference_modes.drop(columns = "weight")
df_reference_modes = pd.DataFrame({ "mode": modes, "weight": df_reference_modes.sum(axis = 0) })
df_reference_modes["data"] = "reference"

df_evaluation_modes = df_evaluation[columns + ["weight"]]
df_evaluation_modes = df_evaluation_modes.rename(columns = rename)
for mode in modes: df_evaluation_modes[mode] *= df_evaluation_modes["weight"].values
df_evaluation_modes = df_evaluation_modes.drop(columns = "weight")
df_evaluation_modes = pd.DataFrame({ "mode": modes, "weight": df_evaluation_modes.sum(axis = 0) })
df_evaluation_modes["data"] = "calibration"

df_initial_modes = df_initial[columns + ["weight"]]
df_initial_modes = df_initial_modes.rename(columns = rename)
for mode in modes: df_initial_modes[mode] *= df_initial_modes["weight"].values
df_initial_modes = df_initial_modes.drop(columns = "weight")
df_initial_modes = pd.DataFrame({ "mode": modes, "weight": df_initial_modes.sum(axis = 0) })
df_initial_modes["data"] = "initial"

df_modes = pd.concat([df_reference_modes, df_evaluation_modes, df_initial_modes])
px.bar(df_modes, x = "mode", y = "weight", color = "data", barmode = "group")